# Ordered Logistic Regression Results for Adoption Predictors: Exploration with `mlcroissant`
This notebook demonstrates how to explore a Croissant-encoded dataset using the `mlcroissant` library. The target dataset contains ordered logistic regression outputs on the adoption of indigenous and modern knowledge in rangeland management in Northern Kenya.

### Dataset Source
The dataset is defined by a Croissant schema and accessible via the following URL:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. Access dataset metadata directly; do not treat it like a dictionary or list.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets and their `@id`s, then drill down to their fields and columns. For all references, use their `@id`.

In [ ]:
# List all record sets by @id
ds = dataset  # Shorthand

record_sets = []
print("Available Record Sets:")
for record_set in ds.record_sets:
    print(f"- {record_set['@id']}: {record_set['name']} ({record_set.get('@type', '')})")
    record_sets.append(record_set['@id'])

print("\nSummary of fields per record set:")
for record_set in ds.record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        # field could be a dict or a reference (string @id)
        if isinstance(field, dict):
            fid = field.get('@id', str(field))
        else:
            fid = field
        print(f"  - Field @id: {fid}")

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for further analysis, referencing by their `@id`. Explore the columns (keys) available.

In [ ]:
# Extract data from the record sets using their @id
if not record_sets:
    print("No record sets defined in the metadata.")
else:
    dataframes = {}
    for rsid in record_sets:
        print(f"\nLoading records from record set @id: {rsid}")
        records = list(ds.records(record_set=rsid))
        if not records:
            print(f"  No records found for {rsid}.")
            continue
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Preview:\n{df.head()}\n")
    # Select the first record set if available for further analysis
    if dataframes:
        chosen_record_set_id = list(dataframes.keys())[0]
        print(f"Sample columns from record set {chosen_record_set_id}:")
        print(dataframes[chosen_record_set_id].columns.tolist())
        display(dataframes[chosen_record_set_id].head())
    else:
        print("No non-empty record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Prepare and process the dataset for downstream analysis.
- Select a numeric field by `@id` for filtering (e.g., coefficient, log likelihood).
- Normalize a numeric field.
- Group by a key field (e.g., knowledge type or gender).
Please adjust `numeric_field_id`, `group_field_id`, and `record_set_id` to match valid `@id`s from the output above.

In [ ]:
# Edit these variables with values from your record set@id and field@id from previous steps:
record_set_id = None
numeric_field_id = None   # Set to the column (field @id) that is numeric
group_field_id = None     # Set to a column (field @id) to group by (e.g., gender, knowledge type)

# If there are dataframes, select the first as default
if 'dataframes' in locals() and dataframes:
    if not record_set_id:
        record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Analysing record set: {record_set_id}")
    print(f"Available fields: {df.columns.tolist()}")
    # Try to infer a likely numeric field
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            numeric_field_id = col
            print(f"Auto-selected numeric field: {numeric_field_id}")
            break
    if not numeric_field_id:
        print("Please manually set numeric_field_id to a numeric column above.")
else:
    print("No record sets loaded; please run the previous step and inspect available fields.")

# EDA: Filter, normalize, and optionally group
if 'df' in locals() and numeric_field_id and numeric_field_id in df.columns:
    threshold = 10  # Example threshold
    filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    if not group_field_id:
        # Try to suggest a likely grouping field
        for col in filtered_df.columns:
            if col != numeric_field_id and filtered_df[col].nunique() < 10:
                group_field_id = col
                print(f"Auto-selected group field: {group_field_id}")
                break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
    else:
        print("No group field set or found.")
else:
    print("Unable to proceed with EDA — please check record set data loaded above.")

## 5. Visualization
Visualize numeric distributions or group differences in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=filtered_df[group_field_id], y=pd.to_numeric(filtered_df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for visualization. Please check previous steps.")

## 6. Conclusion
This notebook demonstrates how to use the `mlcroissant` library to load, inspect, and analyze a dataset described by a Croissant schema. Adjust variable names and field `@id`s as needed for your specific dataset. Further analysis might include statistical testing, more detailed group analyses, or exporting processed data for machine learning workflows.